In [1]:
import seaborn as sns
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor

In [2]:
# "penguins" dataset in seaborn:
# Original Research Paper: 
# Gorman KB, Williams TD, Fraser WR (2014) Ecological Sexual Dimorphism 
# and Environmental Variability within a Community of Antarctic Penguins 
# (Genus Pygoscelis). PLoS ONE 9(3): e90081
# Environmental Data Initiative (EDI): The raw ecological datasets 
# (#219, #220, #221) containing extra variables such as blood isotope 
# ratios and egg clutch observations are hosted directly on the 
# Environmental Data Initiative (EDI) Data Portal under the 
# Palmer Station LTER project.   


# Load Palmer Penguins dataset and drop rows with missing values
df = sns.load_dataset('penguins').dropna()

# Describe the data
df.describe(include='all')


,species,island,bill_length_mm,bill_depth_mm,flipper_length_mm,body_mass_g,sex
count,333,333,333.000000,333.000000,333.000000,333.000000,333
unique,3,3,NaN,NaN,NaN,NaN,2
top,Adelie,Biscoe,NaN,NaN,NaN,NaN,Male
freq,146,163,NaN,NaN,NaN,NaN,168
mean,NaN,NaN,43.992793,17.164865,200.966967,4207.057057,NaN
std,NaN,NaN,5.468668,1.969235,14.015765,805.215802,NaN
min,NaN,NaN,32.100000,13.100000,172.000000,2700.000000,NaN
25%,NaN,NaN,39.500000,15.600000,190.000000,3550.000000,NaN
50%,NaN,NaN,44.500000,17.300000,197.000000,4050.000000,NaN
75%,NaN,NaN,48.600000,18.700000,213.000000,4775.000000,NaN


In [3]:
# What are the different species of penguins?
print(df['species'].unique())

<StringArray>
['Adelie', 'Chinstrap', 'Gentoo']
Length: 3, dtype: str


In [4]:
# Encode categorical features (species, island, sex) and define targets
# Let's say we want to predict body mass from lots of other variables:
X = pd.get_dummies(df[['species', 'island', 'sex', 'bill_length_mm', 'bill_depth_mm', 'flipper_length_mm']], drop_first=True)
y = df['body_mass_g']
print(X.shape)
print(y.shape)

(333, 8)
(333,)


In [5]:
# Train/Test Split & Model Training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [6]:
# Jessie's "quickstats" function, originally written in matlab and translated here.
def quickstats(var1, var2):
    """
    Run basic statistics comparing predicted (var1) vs observed (var2) data.
    """
    var1 = np.asarray(var1).flatten()
    var2 = np.asarray(var2).flatten()
    
    mask = ~np.isnan(var1) & ~np.isnan(var2)
    v1 = var1[mask]
    v2 = var2[mask]
    
    if len(v1) == 0:
        return np.nan, np.nan, np.nan, np.nan, np.nan, np.nan

    bias = np.mean(v1) - np.mean(v2)
    mean_ratio = np.mean(v1) / np.mean(v2)
    mae = np.mean(np.abs(v1 - v2))
    rmse = np.sqrt(np.mean((v1 - v2) ** 2))
    mean_apd = np.mean(np.abs((v1 - v2) / v2))
    
    r_matrix = np.corrcoef(v1, v2)
    r = r_matrix[0, 1] if r_matrix.shape == (2, 2) else np.nan
    
    return bias, mean_ratio, mae, rmse, mean_apd, r

In [7]:
# Evaluate Model Performance
bias, mean_ratio, mae, rmse, mean_apd, r = quickstats(y_pred, y_test)

print("--- Regression Model Evaluation Metrics to Predict Penguin Body Mass ---")
print(f"Bias:                          {bias:.2f} g")
print(f"Mean Ratio (Pred/Obs):         {mean_ratio:.4f}")
print(f"MAE:                           {mae:.2f} g")
print(f"RMSE:                          {rmse:.2f} g")
print(f"Mean APD (Absolute % Diff):    {mean_apd * 100:.2f}%")
print(f"Correlation Coefficient (R):    {r:.4f}")

--- Regression Model Evaluation Metrics to Predict Penguin Body Mass ---
Bias:                          -46.57 g
Mean Ratio (Pred/Obs):         0.9888
MAE:                           237.44 g
RMSE:                          290.91 g
Mean APD (Absolute % Diff):    5.90%
Correlation Coefficient (R):    0.9333


In [ ]:
# Group 1: Classification of different species (supervised)
# Group 2: Clustering (unsupervised)

In [ ]:
# GROUP 1: RANDOM FOREST CLASSIFICATION (SUPERVISED)
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_recall_fscore_support)

# Define input feature matrix X (all available data except species) and target vector y (species).
# I chose to use everything: island, bill_length_mm, bill_depth_mm, flipper_length_mm, body_mass_g, and sex.
X_cls = pd.get_dummies(df.drop(columns=["species"]), drop_first=True)
y_cls = df["species"]

# Train/Test Split ("stratify" ensures equal species representation in train/test)
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42, stratify=y_cls)

# Fit Random Forest Classifier
clf_model = RandomForestClassifier(random_state=42)
clf_model.fit(X_train_cls, y_train_cls)

# Predict species on test set
y_pred_cls = clf_model.predict(X_test_cls)

# Model Evaluation
# Confusion Matrix
cm = confusion_matrix(y_test_cls, y_pred_cls, labels=clf_model.classes_)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {c}" for c in clf_model.classes_],
    columns=[f"Pred {c}" for c in clf_model.classes_])

# Precision, Recall, and F1 Score (weighted average across species)
precision_w, recall_w, f1_w, _ = precision_recall_fscore_support(
    y_test_cls, y_pred_cls, average="weighted")

print("--- Classification Model Evaluation to Predict Penguin Species ---")
print("\nConfusion Matrix:")
print(cm_df)
print(f"\nWeighted Precision : {precision_w:.4f}")
print(f"Weighted Recall    : {recall_w:.4f}")
print(f"Weighted F1 Score  : {f1_w:.4f}")
print("\nDetailed Per-Class Metrics:")
print(classification_report(y_test_cls, y_pred_cls))

In [ ]:
# Classification with Random Forest - how important were the different input variables (features)?
# Access the .feature_importances_ attribute from your trained clf_model from above,
# and pair it with your feature column names (X_cls.columns) using a pandas "Series" sorted in descending order.

# Extract feature importances and map to column names
importances = pd.Series(clf_model.feature_importances_, index=X_cls.columns)

# Sort importances in descending order
importances_sorted = importances.sort_values(ascending=False)
print(importances_sorted)

# Optional: Display as percentages formatted nicely
importances_df = pd.DataFrame(
    { "Feature": importances_sorted.index,
        "Importance": importances_sorted.values,
        "Percentage": (importances_sorted.values * 100).round(2)})
print(importances_df.to_string(index=False))

In [ ]:
# GROUP 2: K-MEANS CLUSTERING (UNSUPERVISED)
from sklearn.cluster import KMeans
from sklearn.metrics import (
    calinski_harabasz_score,
    davies_bouldin_score,
    silhouette_score,)
from sklearn.preprocessing import StandardScaler

# Prepare data without species labels
# This is the same as we did for classification, but we don't name it as the target here.
X_cluster = pd.get_dummies(df.drop(columns=["species"]), drop_first=True)

# Scale features (essential for distance-based clustering like K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_cluster)

# Fit K-Means with k=3 clusters
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

# odel Evaluation Metrics
# Calinski-Harabasz Index (Higher is better)
ch_index = calinski_harabasz_score(X_scaled, cluster_labels)

# Davies-Bouldin Index (Lower is better, 0 is ideal)
db_index = davies_bouldin_score(X_scaled, cluster_labels)

# Silhouette Score (Ranges from -1 to 1; higher is better)
sil_value = silhouette_score(X_scaled, cluster_labels)

# Total Variance:
# - Total dataset variance across scaled features
# - Total within-cluster variance (Inertia is the same thing as within-cluster variance WCSS)
total_data_variance = np.sum(np.var(X_scaled, axis=0))
wcss_inertia = kmeans.inertia_
within_cluster_variance = wcss_inertia / len(X_scaled)
tot_variance_explained = (total_data_variance - within_cluster_variance) / total_data_variance

print("\n--- K-Means Clustering Model Evaluation of Grouping Penguins by the Data Alone ---")
print(f"Calinski-Harabasz (CH) Index : {ch_index:.4f}")
print(f"Davies-Bouldin (DB) Index    : {db_index:.4f}")
print(f"Silhouette Value             : {sil_value:.4f}")
print(f"Total Dataset Variance       : {total_data_variance:.4f}")
print(f"Mean Within-Cluster Variance : {within_cluster_variance:.4f}")
print(f"Percent Variance Explained   : {tot_variance_explained:.1%}")

In [ ]:
# Visualized the clusters in a scatter plot, colored by species:
# I chose bill length and flipper length as my x and y axes to visualize, you could do several more combinations.
sns.scatterplot(
    data=df,
    x="bill_length_mm",
    y="flipper_length_mm",
    hue="species",
    palette="viridis")
plt.title("Penguin Species: Bill Length vs. Flipper Length")
plt.show()